# Video Modality Diagnostics — real model run

Runs the full modality diagnostic (`vmd`) with a **real Hugging Face chat model** on the bundled VideoQA sample.

Runtime: free **T4 GPU** is enough (Runtime → Change runtime type → T4 GPU). ~5 min total with the default 1.5B model.

In [ ]:
# Setup idempotente: clona limpio aunque re-ejecutes la celda
import os, shutil, sys
REPO = "/content/video-modality-diagnostics"
os.chdir("/content")
shutil.rmtree(REPO, ignore_errors=True)
!git clone -q https://github.com/mlahozy21/video-modality-diagnostics.git
os.chdir(REPO)
!pip install -q -e ".[vlm]"
for k in list(sys.modules):
    if k.startswith("vmd"):
        del sys.modules[k]
sys.path.insert(0, f"{REPO}/src")
print("setup OK —", os.getcwd())


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

## Sanity check: offline stub (validates the harness, no download)

In [ ]:
from vmd.backends import OfflineStubBackend
from vmd.data import load_sample
from vmd.diagnose import run_diagnostic
from vmd.report import format_report

items = load_sample()
print(format_report(run_diagnostic(OfflineStubBackend(), items), "offline-stub"))

## Real model diagnostic

Default: `Qwen/Qwen2.5-1.5B-Instruct`. Swap `MODEL` for any HF chat model (e.g. `Qwen/Qwen2.5-3B-Instruct`, `meta-llama/Llama-3.2-1B-Instruct`) to compare how strongly different models rely on the language prior.

In [ ]:
from vmd.backends import HFChatBackend

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
backend = HFChatBackend(MODEL)
result = run_diagnostic(backend, items)
print(format_report(result, backend.name))

## Interpretation

- **collapse gap** close to 0 → the model answers from the question/options alone (modality collapse); large → it genuinely uses the media.
- **modality contribution** — which channel the model relies on (accuracy drop when that channel is removed).
- **vision robustness** — a real model should degrade *gracefully* as the visual evidence is corrupted; a flat curve at full accuracy means it never needed vision.

Paste the resulting table into the README (clearly labelled with the model name) to replace the stub demo numbers with a real measurement.

In [ ]:
# Markdown table ready to paste into the README
r = result
rows = [
    ("Full accuracy", r["acc_full"]),
    ("Blind language prior", r["blind_language_prior"]),
    ("Collapse gap", r["collapse_gap"]),
] + [(f"Contribution: {m}", c) for m, c in r["modality_contribution"].items()]
print(f"| Metric ({MODEL}) | Value |")
print("|---|---:|")
for name, v in rows:
    print(f"| {name} | {v:.3f} |")